<a href="https://colab.research.google.com/github/2303A51552/NLP-3-1/blob/main/PROJECT_Text_Translation(English%20to%20German).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas spacy

In [2]:
import pandas as pd
import spacy

In [3]:
nlp=spacy.load('en_core_web_sm')

In [4]:
df=pd.read_csv('/content/Dataset.csv', encoding='latin-1')

In [5]:
display(df.head())

,"If you listen to English programs on the radio, you can learn English for nothing.","Wenn du englische Programme im Radio hrst, kannst du gratis Englisch lernen."
0,"If you stop and relax, this will relieve the t...",Wenn du einfach mal innehltst und dich entspa...
1,"In order to get to know a person, one merely n...","Um einen Menschen kennenzulernen, braucht man ..."
2,In order to qualify for the homestay you must ...,Um dich fr den Aufenthalt in einer Gastfamili...
3,In order to qualify for the homestay you must ...,Um sich fr den Aufenthalt in einer Gastfamili...
4,It doesn't matter how smart you are. If you do...,"Es spielt keine Rolle, wie klug man ist. Ohne ..."


In [6]:
 df.tail()

,"If you listen to English programs on the radio, you can learn English for nothing.","Wenn du englische Programme im Radio hrst, kannst du gratis Englisch lernen."
1526,If someone who doesn't know your background sa...,"Wenn jemand Fremdes dir sagt, dass du dich wie..."
1527,If someone who doesn't know your background sa...,"Wenn jemand, der nicht weiá, woher man kommt, ..."
1528,It may be impossible to get a completely error...,"Es ist wohl unmglich, einen vollkommen fehler..."
1529,I know that adding sentences only in your nati...,"Ich weiá wohl, dass das ausschlieáliche Beitra..."
1530,Doubtless there exists in this world precisely...,Ohne Zweifel findet sich auf dieser Welt zu je...


In [7]:
import pandas as pd
import re
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
# --Clean and Prepare Text ---
def preprocess_text(text, lang='en'):
    """
    Cleans and preprocesses a single string of text.
    - Lowercases
    - Removes punctuation (keeps basic letters, numbers, and German chars)
    - Normalizes whitespace
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Remove punctuation. Keep letters, numbers, and German characters.
    if lang == 'de':
        text = re.sub(r'[^a-zäöüß0-9\s]', '', text)
    else: # 'en' or default
        text = re.sub(r'[^a-z0-9\s]', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [9]:
# Clean the English column
df['cleaned_english'] = df['If you listen to English programs on the radio, you can learn English for nothing.'].apply(lambda x: preprocess_text(x, lang='en'))

# Clean the German column AND add <sos> (start) and <eos> (end) tokens,essential for training sequence-to-sequence models.
df['cleaned_german'] = df['Wenn du englische Programme im Radio hrst, kannst du gratis Englisch lernen.'].apply(lambda x: f"<sos> {preprocess_text(x, lang='de')} <eos>")

print("Cleaned Data with Tokens:")
print(df.head())
print("-" * 30)

Cleaned Data with Tokens:
  If you listen to English programs on the radio, you can learn English for nothing.  \
0  If you stop and relax, this will relieve the t...                                   
1  In order to get to know a person, one merely n...                                   
2  In order to qualify for the homestay you must ...                                   
3  In order to qualify for the homestay you must ...                                   
4  It doesn't matter how smart you are. If you do...                                   

  Wenn du englische Programme im Radio hrst, kannst du gratis Englisch lernen.  \
0  Wenn du einfach mal innehltst und dich entspa...                              
1  Um einen Menschen kennenzulernen, braucht man ...                              
2  Um dich fr den Aufenthalt in einer Gastfamili...                              
3  Um sich fr den Aufenthalt in einer Gastfamili...                              
4  Es spielt keine Rolle, wie 

In [10]:
# --- 3. Build Vocabularies & Tokenize ---
# We will create two separate "tokenizers" (vocabularies).
# One for English (input) and one for German (target).
input_tokenizer = Tokenizer(filters='', lower=False, oov_token="<unk>")
target_tokenizer = Tokenizer(filters='', lower=False, oov_token="<unk>")

# Build the vocabularies based on the cleaned text
input_tokenizer.fit_on_texts(df['cleaned_english'])
target_tokenizer.fit_on_texts(df['cleaned_german'])

In [11]:
# --- 4. Convert Text to Integer Sequences ---
input_sequences = input_tokenizer.texts_to_sequences(df['cleaned_english'])
target_sequences = target_tokenizer.texts_to_sequences(df['cleaned_german'])

In [12]:
# --- 5. Pad Sequences ---
# Models need inputs to be the same length. We'll pad them.
# 'post' means padding is added at the end of the sequence.
input_sequences_padded = pad_sequences(input_sequences, padding='post')
target_sequences_padded = pad_sequences(target_sequences, padding='post')

# Get the maximum sequence lengths
max_input_len = input_sequences_padded.shape[1]
max_target_len = target_sequences_padded.shape[1]

# Get the size of each vocabulary
input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

print(f"Total English Vocabulary Size: {input_vocab_size}")
print(f"Total German Vocabulary Size: {target_vocab_size}")
print(f"Max Input Sequence Length: {max_input_len}")
print(f"Max Target Sequence Length: {max_target_len}")
print("-" * 30)

# --- Show a sample of the final preprocessed data ---
print("Sample of Final Processed Data (first 5 pairs):\n")
for i in range(len(df)): # Iterate up to the actual number of rows
    if i >= 5: # Limit to the first 5 pairs if the dataframe is large
        break
    print(f"Pair {i+1}")
    print(f"  English (Original): {df['If you listen to English programs on the radio, you can learn English for nothing.'].iloc[i]}")
    print(f"  German (Original):  {df['Wenn du englische Programme im Radio hrst, kannst du gratis Englisch lernen.'].iloc[i]}")
    print(f"  English (Cleaned):  {df['cleaned_english'].iloc[i]}")
    print(f"  German (Cleaned):   {df['cleaned_german'].iloc[i]}")
    print(f"  English (Sequence): {input_sequences[i]}")
    print(f"  German (Sequence):  {target_sequences[i]}")
    print(f"  English (Padded):   {input_sequences_padded[i]}")
    print(f"  German (Padded):   {target_sequences_padded[i]}")

Total English Vocabulary Size: 3969
Total German Vocabulary Size: 5467
Max Input Sequence Length: 101
Max Target Sequence Length: 77
------------------------------
Sample of Final Processed Data (first 5 pairs):

Pair 1
  English (Original): If you stop and relax, this will relieve the tension and stress in your shoulders.
  German (Original):  Wenn du einfach mal innehltst und dich entspannst, wird die Verspannung in deinen Schultern geringer werden.
  English (Cleaned):  if you stop and relax this will relieve the tension and stress in your shoulders
  German (Cleaned):   <sos> wenn du einfach mal innehltst und dich entspannst wird die verspannung in deinen schultern geringer werden <eos>
  English (Sequence): [27, 12, 417, 7, 1206, 51, 75, 1856, 2, 1857, 7, 1858, 9, 47, 1207]
  German (Sequence):  [2, 29, 27, 201, 221, 1971, 9, 97, 1972, 55, 5, 1973, 12, 556, 1974, 1187, 67, 3]
  English (Padded):   [  27   12  417    7 1206   51   75 1856    2 1857    7 1858    9   47
 1207    0  

In [13]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.utils import to_categorical

# --- 1. Prepare Data for Encoder-Decoder Model ---
# We need to re-structure the data slightly for the Seq2Seq model.
# The decoder needs two versions of the target data:
# 1. decoder_input_data: The target sequence (German)
# 2. decoder_target_data: The target sequence shifted one step ahead.

decoder_input_data = target_sequences_padded

# decoder_target_data is the same, but shifted one step to the left.
# [ Hallo, wie, gehts, <eos>, <pad>, <pad>, <pad> ]
# We also remove the first token from the original to match the length.
decoder_target_data_temp = np.zeros_like(decoder_input_data)
for i, seq in enumerate(target_sequences_padded):
    decoder_target_data_temp[i, :-1] = seq[1:]



In [14]:
# The 'decoder_target_data' must be one-hot encoded,
# because we are predicting one word out of the entire vocabulary.
# This can consume a lot of memory if the vocab is large.
decoder_target_data = to_categorical(decoder_target_data_temp, num_classes=target_vocab_size)

print("Shape of encoder input data:", input_sequences_padded.shape)
print("Shape of decoder input data:", decoder_input_data.shape)
print("Shape of decoder target data:", decoder_target_data.shape)
print("-" * 30)

Shape of encoder input data: (1531, 101)
Shape of decoder input data: (1531, 77)
Shape of decoder target data: (1531, 77, 5467)
------------------------------


In [15]:
# --- 2. Define the Model Architecture ---
# We'll use the "Functional API" from Keras.

# Latent dimension (hidden size) of the LSTM layers
latent_dim = 256
embedding_dim = 100

# --- Encoder ---
# Takes the English sequence as input.
encoder_inputs = Input(shape=(max_input_len,), name='encoder_input')

# Embedding layer
# Turns positive integers (indexes) into dense vectors of fixed size.
enc_embedding_layer = Embedding(input_vocab_size, embedding_dim, mask_zero=True)
enc_emb = enc_embedding_layer(encoder_inputs)

**LSTM**


In [16]:
# The LSTM layer.
# 'return_state=True' tells the LSTM to return its final hidden state
# and cell state, which will be the "context" for the decoder.
encoder_lstm = LSTM(latent_dim, return_state=True, name='encoder_lstm')
# We don't care about the encoder 'outputs', just the states.
_, state_h, state_c = encoder_lstm(enc_emb)

# We package the final states into a list. This is the 'context'.
encoder_states = [state_h, state_c]


# --- Decoder ---
# Takes the German sequence as input, plus the encoder's context.
decoder_inputs = Input(shape=(max_target_len,), name='decoder_input')

# Embedding layer
dec_embedding_layer = Embedding(target_vocab_size, embedding_dim, mask_zero=True)
dec_emb = dec_embedding_layer(decoder_inputs)

# The Decoder LSTM.
# 'return_sequences=True' makes it output a full sequence (not just the last output).
# 'return_state=True' is needed for inference later, but we ignore it during training.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name='decoder_lstm')

# The key step: We initialize the decoder's state
# with the encoder's final states ('encoder_states').
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# A 'Dense' layer (fully-connected) to predict the next word.
# It has 'target_vocab_size' neurons and a 'softmax' activation
# to output a probability distribution over the entire German vocabulary.
decoder_dense = Dense(target_vocab_size, activation='softmax', name='decoder_dense')
decoder_outputs = decoder_dense(decoder_outputs)


In [17]:

# --- 3. Build the final "Training Model" ---
# This model maps the encoder's input and the decoder's input
# to the decoder's target (the one-hot encoded, shifted predictions).
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [18]:
# --- 4. Compile the Model ---
# We use 'categorical_crossentropy' because our target is one-hot encoded.
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print a summary of the model
model.summary()
print("-" * 30)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 101)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 77)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 101, 100)  │    396,900 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 101)       │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 77, 100)   │    546,700 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    365,568 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 77, 256), │    365,568 │ embedding_1[0][0… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 77, 5467)  │  1,405,019 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,079,755 (11.75 MB)

 Trainable params: 3,079,755 (11.75 MB)

 Non-trainable params: 0 (0.00 B)

------------------------------


In [22]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# --- 1. Define and Load Data ---
# Create a dummy DataFrame with the column names expected in the original script
# Note: The original script loads a file named 'Dataset.csv'.
# This reduced version creates a minimal DataFrame in memory for execution.
data = {
    'english': ['She speaks English.', 'I am learning German.'],
    'german': ['Sie spricht Englisch.', 'Ich lerne Deutsch.']
}
df = pd.DataFrame(data) # Directly create DataFrame with desired columns and data

# --- 2. Preprocess Data ---
df['cleaned_german'] = df['german'].apply(lambda x: '<sos> ' + x + ' <eos>')
df['cleaned_english'] = df['english']

# Tokenization
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(df['cleaned_english'])
eng_vocab_size = len(eng_tokenizer.word_index) + 1

ger_tokenizer = Tokenizer()
ger_tokenizer.fit_on_texts(df['cleaned_german'])
ger_vocab_size = len(ger_tokenizer.word_index) + 1

# Convert text to sequences
input_sequences = eng_tokenizer.texts_to_sequences(df['cleaned_english'])
target_sequences = ger_tokenizer.texts_to_sequences(df['cleaned_german'])

# Padding
max_eng_len = max(len(s) for s in input_sequences)
max_ger_len = max(len(s) for s in target_sequences)

input_sequences_padded = pad_sequences(input_sequences, maxlen=max_eng_len, padding='post')
target_sequences_padded = pad_sequences(target_sequences, maxlen=max_ger_len, padding='post')

# Create target data for training (one-hot encoded, shifted target sequence)
decoder_target_data = np.zeros(
    (len(df), max_ger_len, ger_vocab_size),
    dtype='float32'
)

for i, seq in enumerate(target_sequences_padded):
    for t, word_index in enumerate(seq):
        if t > 0 and t < max_ger_len:
            decoder_target_data[i, t-1, word_index] = 1.0

# Store reverse mapping for decoding
rev_ger_word_index = dict((i, word) for word, i in ger_tokenizer.word_index.items())

# --- 3. Define and Train Seq2Seq Model ---
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(max_eng_len,))
encoder_embedding = tf.keras.layers.Embedding(eng_vocab_size, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_ger_len,))
decoder_embedding = tf.keras.layers.Embedding(ger_vocab_size, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(ger_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Model Definition and Training
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model.fit(
    [input_sequences_padded, target_sequences_padded],
    decoder_target_data,
    batch_size=1,
    epochs=10,
    validation_split=0.0,
    verbose=0 # Suppress training output
)

# --- 4. Inference Model (Decoding Setup) ---
# Encoder (Inference)
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder (Inference)
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding, initial_state=decoder_states_inputs
)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

# --- 5. The _decode_sequence Function ---
def _decode_sequence(input_seq, local_encoder_model, local_decoder_model, local_ger_tokenizer, local_rev_ger_word_index, local_max_ger_len):
    # Encode the input sequence
    states_value = local_encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence for the start token.
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = local_ger_tokenizer.word_index['sos']

    # Sampling loop
    decoded_sentence = ''
    stop_condition = False
    while not stop_condition:
        output_tokens, h, c = local_decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        # Sample a token index
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = local_rev_ger_word_index.get(sampled_token_index, '')

        # Exit condition
        if sampled_word == 'eos' or len(decoded_sentence.split()) > local_max_ger_len:
            stop_condition = True
        elif sampled_word != 'sos':
            decoded_sentence += ' ' + sampled_word

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return decoded_sentence.strip()

# --- 6. Calculate BLEU Score ---
chencherry = SmoothingFunction().method1
all_bleu_scores = []
num_samples_to_test = len(df)

for i in range(num_samples_to_test):
    input_seq = input_sequences_padded[i:i+1]

    predicted_sentence = _decode_sequence(
        input_seq,
        encoder_model,
        decoder_model,
        ger_tokenizer,
        rev_ger_word_index,
        max_ger_len
    )

    reference_sentence = df['cleaned_german'].iloc[i]
    reference_sentence = reference_sentence.replace('<sos>', '').replace('<eos>', '').strip()

    # Split sentences into lists of words for NLTK
    reference_tokens = [reference_sentence.split()]
    predicted_tokens = predicted_sentence.split()

    if predicted_tokens:
        score = sentence_bleu(
            reference_tokens,
            predicted_tokens,
            smoothing_function=chencherry
        )
    else:
        score = 0.0

    all_bleu_scores.append(score)

# Calculate and print the average BLEU score
average_bleu = np.mean(all_bleu_scores)
print(f"Average BLEU Score: **{average_bleu:.4f}**")

Average BLEU Score: **0.0453**


**Word2Vec**

In [23]:
# First, you need to install gensim
!pip install gensim

import pandas as pd
import re
from gensim.models import Word2Vec
import numpy as np

# --- 1. Load and Clean Data (from your preprocessing script) ---
# (Assuming 'df' with 'cleaned_english' is in memory)
# We need a list of sentences, where each sentence is a list of words.
english_sentences = [row.split() for row in df['cleaned_english'] if row]

# --- 2. Train Word2Vec Model ---
print("Training Word2Vec model on English text...")

# embedding_dim MUST match the 'embedding_dim' you plan to use in your Keras model
embedding_dim = 100

w2v_model = Word2Vec(sentences=english_sentences,
                     vector_size=embedding_dim,
                     window=5,
                     min_count=1,
                     workers=4)

print("Word2Vec training complete.")

# --- 3. Create Weight Matrix for Keras ---
# We need the 'input_tokenizer' from your preprocessing script
# (Assuming 'input_tokenizer' is in memory)
input_vocab_size = len(input_tokenizer.word_index) + 1

# Create an empty matrix to hold the weights
embedding_matrix = np.zeros((input_vocab_size, embedding_dim))

# Fill the matrix with the vectors from Word2Vec
for word, i in input_tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print("Created embedding matrix of shape:", embedding_matrix.shape)

# --- 4. Use this Matrix in your Keras Model ---
# Now, when you build your Encoder, you replace the old
# 'enc_embedding_layer' with this new one:

# The OLD layer was:
# enc_embedding_layer = Embedding(input_vocab_size, embedding_dim, mask_zero=True)

# The NEW layer is:
enc_embedding_layer = Embedding(
    input_vocab_size,
    embedding_dim,
    weights=[embedding_matrix],  # <-- 1. Load the pre-trained weights
    trainable=False,             # <-- 2. Freeze the layer (so it doesn't change)
    mask_zero=True
)

# Now, build the rest of your Encoder and Decoder exactly as before,
# starting with this new 'enc_embedding_layer'.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 20.3 MB/s eta 0:00:00
Training Word2Vec model on English text...
Word2Vec training complete.
Created embedding matrix of shape: (3969, 100)


In [24]:
# First, install the transformers library and PyTorch (or TensorFlow)
!pip install transformers
!pip install torch
# or !pip install tensorflow

**Direct Translation**

In [25]:
from transformers import pipeline

# --- 1. Load the Pre-trained Translation Model ---
# We'll use a model from "Helsinki-NLP" built for this exact task.
# The 'pipeline' function handles everything:
# tokenization, model loading, and decoding.
print("Loading translation model...")
translator = pipeline(
    "translation_en_to_de",                # The task
    model="Helsinki-NLP/opus-mt-en-de"     # The specific model
)
print("Model loaded.")

# --- 2. Translate Sentences ---
english_sentences = [
    "Hello, how are you today?",
    "A cat sat on the mat.",
    "This is a powerful model."
]

print("\nTranslating...")
german_translations = translator(english_sentences)

Loading translation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

...

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


Model loaded.

Translating...


In [26]:
# --- 3. View Results ---
for i, result in enumerate(german_translations):
    print(f"\nEnglish: {english_sentences[i]}")
    print(f"German:  {result['translation_text']}")


English: Hello, how are you today?
German:  Hallo, wie geht's dir heute?

English: A cat sat on the mat.
German:  Eine Katze saß auf der Matte.

English: This is a powerful model.
German:  Das ist ein mächtiges Modell.
